# rabi prefix sensitivity vs shots
See the repository README for the paper configurations and cached-data reproduction commands.


In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import hashlib
import json
import math
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from scipy.stats import chi2, ncx2, norm, poisson


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "src" / "dm_search").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing src/dm_search")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from dm_search.rabi import (  # noqa: E402
    RabiExperimentConfig,
    RabiParticleSpec,
    RabiReadoutConfig,
    build_rabi_particle_grid,
    longitudinal_relaxation_factor,
    rabi_oracle_known_signal_power,
    sample_rabi_log_bayes_factors_at_checkpoints,
    transverse_relaxation_factor,
    with_true_phase_amplitude,
)
from dm_search.statistics import null_threshold  # noqa: E402


plt.style.use("seaborn-v0_8-whitegrid")

# Shared manuscript typography and frameless legends.
plt.style.use(REPO_ROOT / "plot_final" / "rabi_paper.mplstyle")


In [ ]:
def parse_int_list(text: str) -> np.ndarray:
    return np.asarray([int(piece.strip()) for piece in text.split(",") if piece.strip()], dtype=int)


def parse_float_list(text: str) -> np.ndarray:
    return np.asarray([float(piece.strip()) for piece in text.split(",") if piece.strip()], dtype=float)


def default_checkpoints(n_shot_max: int) -> np.ndarray:
    base = np.asarray(
        [
            32,
            48,
            64,
            96,
            128,
            192,
            256,
            384,
            512,
            768,
            1024,
            1536,
            2048,
            3072,
            4096,
            6144,
            8192,
            12288,
            16384,
            24576,
            32768,
            49152,
            65536,
        ],
        dtype=int,
    )
    checkpoints = base[base <= int(n_shot_max)]
    if checkpoints.size == 0 or checkpoints[-1] != int(n_shot_max):
        checkpoints = np.asarray(sorted(set([*checkpoints.tolist(), int(n_shot_max)])), dtype=int)
    return checkpoints


RUN_PRESET = os.environ.get("RABI_PREFIX_PRESET", "final")
PRESETS = {
    "quick": dict(n_shot_max=256, n_experiments=24, batch_size=12, phi_r_list="0.06,0.10,0.16,0.25,0.38", target_powers="0.5,0.8", amplitude_count=12, drive_phase_count=24, candidate_angle_count=8, candidate_polar_count=5),
    "draft": dict(n_shot_max=1024, n_experiments=100, batch_size=128, phi_r_list="0.025,0.035,0.05,0.065,0.08,0.10,0.12,0.15,0.19,0.24,0.32", target_powers="0.5,0.7,0.9", amplitude_count=24, drive_phase_count=48, candidate_angle_count=16, candidate_polar_count=5),
    "final": dict(n_shot_max=16384, n_experiments=1000, batch_size=512, phi_r_list="0.025,0.035,0.05,0.065,0.08,0.10,0.12,0.15,0.19,0.24,0.32", target_powers="0.5,0.7,0.9", amplitude_count=48, drive_phase_count=96, candidate_angle_count=16, candidate_polar_count=7),
}
if RUN_PRESET not in PRESETS:
    raise ValueError(f"unknown RABI_PREFIX_PRESET={RUN_PRESET!r}")
P = PRESETS[RUN_PRESET]

ALPHA = float(os.environ.get("RABI_PREFIX_ALPHA", "0.05"))
N_SHOT_MAX = int(os.environ.get("RABI_PREFIX_N_SHOT_MAX", P["n_shot_max"]))
N_EXPERIMENTS = int(os.environ.get("RABI_PREFIX_N_EXPERIMENTS", P["n_experiments"]))
N_NULL = int(os.environ.get("RABI_PREFIX_N_NULL", N_EXPERIMENTS))
N_SIGNAL = int(os.environ.get("RABI_PREFIX_N_SIGNAL", N_EXPERIMENTS))
BATCH_SIZE = int(os.environ.get("RABI_PREFIX_BATCH_SIZE", P["batch_size"]))
PHASE_GRID = parse_float_list(os.environ.get("RABI_PREFIX_PHI_R_LIST", P["phi_r_list"]))
TARGET_POWERS = tuple(parse_float_list(os.environ.get("RABI_PREFIX_TARGET_POWERS", P["target_powers"])).tolist())
checkpoint_text = os.environ.get("RABI_PREFIX_CHECKPOINTS", "")
CHECKPOINTS = parse_int_list(checkpoint_text) if checkpoint_text.strip() else default_checkpoints(N_SHOT_MAX)
CHECKPOINTS = np.asarray(sorted(set(int(x) for x in CHECKPOINTS if 0 <= int(x) <= N_SHOT_MAX)), dtype=int)
if CHECKPOINTS.size == 0 or CHECKPOINTS[-1] != N_SHOT_MAX:
    CHECKPOINTS = np.asarray(sorted(set([*CHECKPOINTS.tolist(), N_SHOT_MAX])), dtype=int)

SENSING_TIME = float(os.environ.get("RABI_PREFIX_SENSING_TIME", "1.0"))
T1_TIME = float(os.environ.get("RABI_PREFIX_T1", str(SENSING_TIME)))
T2_TIME = float(os.environ.get("RABI_PREFIX_T2", str(SENSING_TIME)))
DEAD_TIME = float(os.environ.get("RABI_PREFIX_DEAD_TIME", "0.0"))
CONTRAST = float(os.environ.get("RABI_PREFIX_CONTRAST", "0.999"))
READOUT_ERROR = float(os.environ.get("RABI_PREFIX_READOUT_ERROR", "0.0005"))

PHASE_AMPLITUDE_MIN = float(os.environ.get("RABI_PREFIX_AMPLITUDE_MIN", "0.02"))
PHASE_AMPLITUDE_MAX = float(os.environ.get("RABI_PREFIX_AMPLITUDE_MAX", "1.2"))
PHASE_AMPLITUDE_COUNT = int(os.environ.get("RABI_PREFIX_AMPLITUDE_COUNT", P["amplitude_count"]))
DETUNING_MIN = float(os.environ.get("RABI_PREFIX_DETUNING_MIN", "0.0"))
DETUNING_MAX = float(os.environ.get("RABI_PREFIX_DETUNING_MAX", "0.0"))
DETUNING_COUNT = int(os.environ.get("RABI_PREFIX_DETUNING_COUNT", "0"))
DRIVE_PHASE_COUNT = int(os.environ.get("RABI_PREFIX_DRIVE_PHASE_COUNT", P["drive_phase_count"]))
CANDIDATE_ANGLE_COUNT = int(os.environ.get("RABI_PREFIX_CANDIDATE_ANGLE_COUNT", P["candidate_angle_count"]))
CANDIDATE_POLAR_COUNT = int(os.environ.get("RABI_PREFIX_CANDIDATE_POLAR_COUNT", P["candidate_polar_count"]))
PERIODIC_HYBRID_PERIOD = int(os.environ.get("RABI_PREFIX_PERIODIC_HYBRID_PERIOD", "8"))
ADAPTIVE_SAMPLER_STRATEGY = os.environ.get("RABI_PREFIX_ADAPTIVE_STRATEGY", f"periodic_hybrid_sphere_continuous_{PERIODIC_HYBRID_PERIOD}")
PHASE_AVERAGE_MODE = os.environ.get("RABI_PREFIX_PHASE_MODE", "random_per_experiment")

STRATEGIES = {
    "fixed_z": {"label": r"Fixed $z$", "color": "#4C78A8", "marker": "o"},
    "fixed_x": {"label": r"Fixed $x$", "color": "#F58518", "marker": "D"},
    "two_axis": {"label": r"Fixed $x/y$", "color": "#B279A2", "marker": "v"},
    "adaptive": {"label": "Adaptive sphere", "color": "#54A24B", "marker": "s", "sampler_strategy": ADAPTIVE_SAMPLER_STRATEGY},
}
ORACLE_KEY = "oracle_known_signal"
ORACLE_META = {"label": "Global oracle", "color": "black", "marker": "^"}
SENSITIVITY_STRATEGIES = {**STRATEGIES, ORACLE_KEY: ORACLE_META}

base_template = RabiExperimentConfig(
    n_shots=N_SHOT_MAX,
    sensing_time=SENSING_TIME,
    dead_time=DEAD_TIME,
    true_phase_amplitude=float(PHASE_GRID[0]),
    true_detuning=0.0,
    true_drive_phase=0.0,
    readout=RabiReadoutConfig(contrast=CONTRAST, readout_error=READOUT_ERROR, t1=T1_TIME, t2=T2_TIME),
    particles=RabiParticleSpec(
        phase_amplitude_min=PHASE_AMPLITUDE_MIN,
        phase_amplitude_max=PHASE_AMPLITUDE_MAX,
        phase_amplitude_count=PHASE_AMPLITUDE_COUNT,
        detuning_min=DETUNING_MIN,
        detuning_max=DETUNING_MAX,
        detuning_count=DETUNING_COUNT,
        drive_phase_count=DRIVE_PHASE_COUNT,
    ),
    candidate_angles=tuple(np.linspace(0.0, math.pi, CANDIDATE_ANGLE_COUNT, endpoint=False)),
    candidate_polars=tuple(np.linspace(0.0, math.pi, CANDIDATE_POLAR_COUNT)),
    seed=12345,
)

particle_grid = build_rabi_particle_grid(base_template.particles)
TRUE_DRIVE_PHASES = particle_grid.drive_phase_values

OUTPUT_BASENAME = "rabi_prefix_sensitivity_vs_shots"
CACHE_DIR = REPO_ROOT / "plot_final" / "cache"
FIGURE_DIR = REPO_ROOT / "plot_final" / "figure"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
USE_CACHE = os.environ.get("RABI_PREFIX_USE_CACHE", "1") != "0"
FORCE_RECOMPUTE = os.environ.get("RABI_PREFIX_FORCE_RECOMPUTE", "0") == "1"

setup = {
    "preset": RUN_PRESET,
    "alpha": ALPHA,
    "n_shot_max": N_SHOT_MAX,
    "checkpoints": CHECKPOINTS.tolist(),
    "n_null": N_NULL,
    "n_signal": N_SIGNAL,
    "batch_size": BATCH_SIZE,
    "phase_grid": PHASE_GRID.tolist(),
    "target_powers": list(TARGET_POWERS),
    "sensing_time": SENSING_TIME,
    "t1_time": T1_TIME,
    "t2_time": T2_TIME,
    "dead_time": DEAD_TIME,
    "contrast": CONTRAST,
    "readout_error": READOUT_ERROR,
    "particle_shape": particle_grid.shape,
    "candidate_angle_count": len(base_template.candidate_angles),
    "candidate_polar_count": len(base_template.candidate_polars),
    "adaptive_sampler_strategy": ADAPTIVE_SAMPLER_STRATEGY,
    "periodic_hybrid_period": PERIODIC_HYBRID_PERIOD,
    "phase_average_mode": PHASE_AVERAGE_MODE,
    "cache_version": "prefix-checkpoint-v3-fixed-transverse",
}
print(json.dumps(setup, indent=2))


In [ ]:
def stable_hash(payload: dict) -> str:
    blob = json.dumps(payload, sort_keys=True, default=list).encode("utf-8")
    return hashlib.sha256(blob).hexdigest()[:16]


def cache_path() -> Path:
    return CACHE_DIR / f"{OUTPUT_BASENAME}_{RUN_PRESET}_{stable_hash(setup)}.npz"


def power_and_error(signal_statistics: np.ndarray, threshold: float) -> tuple[float, float]:
    power = float(np.mean(signal_statistics >= threshold))
    error = float(np.sqrt(max(power * (1.0 - power), 0.0) / len(signal_statistics)))
    return power, error


def sensitivity_crossing(phases: np.ndarray, powers: np.ndarray, target: float) -> float:
    monotone = np.maximum.accumulate(np.asarray(powers, dtype=float))
    phases = np.asarray(phases, dtype=float)
    if monotone[-1] < target:
        return float("nan")
    index = int(np.searchsorted(monotone, target, side="left"))
    if index == 0:
        return float(phases[0])
    x0, x1 = phases[index - 1], phases[index]
    y0, y1 = monotone[index - 1], monotone[index]
    if y1 <= y0:
        return float(x1)
    return float(x0 + (target - y0) * (x1 - x0) / (y1 - y0))


def h1_phase_averaged_checkpoint_samples(
    config: RabiExperimentConfig,
    strategy: str,
    phase_amplitude: float,
    total_count: int,
    seed_base: int,
) -> np.ndarray:
    if strategy == "fixed_z":
        fixed_config = replace(with_true_phase_amplitude(config, phase_amplitude), true_drive_phase=0.0)
        return sample_rabi_log_bayes_factors_at_checkpoints(
            fixed_config,
            strategy,
            "H1",
            total_count,
            CHECKPOINTS,
            batch_size=BATCH_SIZE,
            seed=seed_base,
            method="numba",
        )

    sampler_strategy = STRATEGIES[strategy].get("sampler_strategy", strategy)
    if PHASE_AVERAGE_MODE == "random_per_experiment":
        phase_rng = np.random.default_rng(seed_base + 91_337)
        random_true_drive_phases = phase_rng.uniform(0.0, 2.0 * np.pi, total_count)
        adaptive_config = replace(with_true_phase_amplitude(config, phase_amplitude), true_drive_phase=0.0)
        return sample_rabi_log_bayes_factors_at_checkpoints(
            adaptive_config,
            sampler_strategy,
            "H1",
            total_count,
            CHECKPOINTS,
            batch_size=BATCH_SIZE,
            seed=seed_base,
            method="numba",
            true_drive_phases=random_true_drive_phases,
        )

    per_phase = int(np.ceil(total_count / len(TRUE_DRIVE_PHASES)))
    pieces = []
    for phase_index, drive_phase in enumerate(TRUE_DRIVE_PHASES):
        adaptive_config = replace(
            with_true_phase_amplitude(config, phase_amplitude),
            true_drive_phase=float(drive_phase),
        )
        pieces.append(
            sample_rabi_log_bayes_factors_at_checkpoints(
                adaptive_config,
                sampler_strategy,
                "H1",
                per_phase,
                CHECKPOINTS,
                batch_size=min(BATCH_SIZE, per_phase),
                seed=seed_base + phase_index,
                method="numba",
            )
        )
    return np.vstack(pieces)[:total_count]


def run_checkpoint_scan() -> tuple[list[dict], list[dict], list[dict], list[dict]]:
    power_rows = []
    sensitivity_rows = []
    oracle_power_rows = []
    oracle_sensitivity_rows = []
    start = time.perf_counter()
    config = replace(base_template, n_shots=N_SHOT_MAX)

    thresholds_by_strategy: dict[str, np.ndarray] = {}
    for strategy_index, strategy in enumerate(STRATEGIES):
        sampler_strategy = STRATEGIES[strategy].get("sampler_strategy", strategy)
        null_samples = sample_rabi_log_bayes_factors_at_checkpoints(
            config,
            sampler_strategy,
            "H0",
            N_NULL,
            CHECKPOINTS,
            batch_size=BATCH_SIZE,
            seed=10_000 + 100 * strategy_index,
            method="numba",
        )
        thresholds = np.asarray([null_threshold(null_samples[:, j], ALPHA) for j in range(CHECKPOINTS.size)], dtype=float)
        thresholds_by_strategy[strategy] = thresholds

        power_by_phase = []
        for phase_index, phase_amplitude in enumerate(PHASE_GRID):
            signal_samples = h1_phase_averaged_checkpoint_samples(
                config,
                strategy,
                float(phase_amplitude),
                N_SIGNAL,
                seed_base=20_000 + 10_000 * strategy_index + 100 * phase_index,
            )
            powers = []
            for checkpoint_index, n_shots in enumerate(CHECKPOINTS):
                power, power_error = power_and_error(signal_samples[:, checkpoint_index], thresholds[checkpoint_index])
                powers.append(power)
                power_rows.append(
                    {
                        "n_shots": int(n_shots),
                        "strategy": strategy,
                        "phase_amplitude": float(phase_amplitude),
                        "power": power,
                        "power_se": power_error,
                        "beta": 1.0 - power,
                        "threshold": float(thresholds[checkpoint_index]),
                        "empirical_alpha": float(np.mean(null_samples[:, checkpoint_index] >= thresholds[checkpoint_index])),
                    }
                )
            power_by_phase.append(powers)
        power_by_phase = np.asarray(power_by_phase, dtype=float)

        for checkpoint_index, n_shots in enumerate(CHECKPOINTS):
            checkpoint_powers = power_by_phase[:, checkpoint_index]
            for target in TARGET_POWERS:
                sensitivity_rows.append(
                    {
                        "n_shots": int(n_shots),
                        "strategy": strategy,
                        "target_power": float(target),
                        "phase_sensitivity": sensitivity_crossing(PHASE_GRID, checkpoint_powers, float(target)),
                    }
                )
        print(f"{strategy}: completed prefix scan", flush=True)

    oracle_phase_grid = np.linspace(max(0.001, 0.5 * float(PHASE_GRID.min())), float(PHASE_GRID.max()), 240)
    for n_shots in CHECKPOINTS:
        oracle_config = replace(base_template, n_shots=int(n_shots))
        oracle_powers = np.asarray(rabi_oracle_known_signal_power(oracle_config, oracle_phase_grid, ALPHA), dtype=float)
        for phase_amplitude, power in zip(oracle_phase_grid, oracle_powers):
            oracle_power_rows.append(
                {
                    "n_shots": int(n_shots),
                    "strategy": ORACLE_KEY,
                    "phase_amplitude": float(phase_amplitude),
                    "power": float(power),
                    "power_se": 0.0,
                }
            )
        for target in TARGET_POWERS:
            oracle_sensitivity_rows.append(
                {
                    "n_shots": int(n_shots),
                    "strategy": ORACLE_KEY,
                    "target_power": float(target),
                    "phase_sensitivity": sensitivity_crossing(oracle_phase_grid, oracle_powers, float(target)),
                }
            )

    elapsed = time.perf_counter() - start
    print(f"elapsed_seconds={elapsed:.1f}")
    return power_rows, sensitivity_rows, oracle_power_rows, oracle_sensitivity_rows


def save_scan(path: Path, power_rows, sensitivity_rows, oracle_power_rows, oracle_sensitivity_rows) -> None:
    np.savez_compressed(
        path,
        setup=json.dumps(setup),
        power_rows=json.dumps(power_rows),
        sensitivity_rows=json.dumps(sensitivity_rows),
        oracle_power_rows=json.dumps(oracle_power_rows),
        oracle_sensitivity_rows=json.dumps(oracle_sensitivity_rows),
    )


def load_scan(path: Path):
    payload = np.load(path, allow_pickle=True)
    return (
        json.loads(str(payload["power_rows"])),
        json.loads(str(payload["sensitivity_rows"])),
        json.loads(str(payload["oracle_power_rows"])),
        json.loads(str(payload["oracle_sensitivity_rows"])),
    )


In [ ]:
path = cache_path()
if USE_CACHE and path.exists() and not FORCE_RECOMPUTE:
    print(f"loading {path}")
    power_rows, sensitivity_rows, oracle_power_rows, oracle_sensitivity_rows = load_scan(path)
else:
    power_rows, sensitivity_rows, oracle_power_rows, oracle_sensitivity_rows = run_checkpoint_scan()
    if USE_CACHE:
        save_scan(path, power_rows, sensitivity_rows, oracle_power_rows, oracle_sensitivity_rows)
        print(f"saved {path}")

combined_power_rows = power_rows + oracle_power_rows
combined_sensitivity_rows = sensitivity_rows + oracle_sensitivity_rows
power_df = pd.DataFrame(combined_power_rows)
sensitivity_df = pd.DataFrame(combined_sensitivity_rows)
display(power_df.head())
display(sensitivity_df.head())


In [ ]:
def selected_plot_checkpoints(checkpoints: np.ndarray, max_count: int = 8) -> np.ndarray:
    if checkpoints.size <= max_count:
        return checkpoints
    indices = np.unique(np.round(np.linspace(0, checkpoints.size - 1, max_count)).astype(int))
    return checkpoints[indices]


NULL_CALIBRATION_TOLERANCE = max(
    0.02,
    4.0 * math.sqrt(ALPHA * (1.0 - ALPHA) / N_NULL),
)


def null_calibration_mask(rows: pd.DataFrame) -> np.ndarray:
    if "empirical_alpha" not in rows:
        return np.ones(len(rows), dtype=bool)
    empirical_alpha = rows["empirical_alpha"].to_numpy(dtype=float)
    return np.isfinite(empirical_alpha) & (np.abs(empirical_alpha - ALPHA) <= NULL_CALIBRATION_TOLERANCE)


calibration_summary = (
    power_df[power_df["strategy"].isin(STRATEGIES)]
    .groupby(["n_shots", "strategy"], as_index=False)["empirical_alpha"]
    .first()
)
uncalibrated_summary = calibration_summary[
    np.abs(calibration_summary["empirical_alpha"] - ALPHA) > NULL_CALIBRATION_TOLERANCE
]
if not uncalibrated_summary.empty:
    print("omitting checkpoints with uncalibrated discrete null thresholds:")
    display(uncalibrated_summary)


plot_checkpoints = selected_plot_checkpoints(CHECKPOINTS)
n_columns = min(4, plot_checkpoints.size)
n_rows = int(np.ceil(plot_checkpoints.size / n_columns))
fig, axes = plt.subplots(n_rows, n_columns, figsize=(3.8 * n_columns, 3.2 * n_rows), sharex=True, sharey=True)
axes = np.asarray(axes).ravel()

for ax, n_shots in zip(axes, plot_checkpoints):
    for strategy, meta in STRATEGIES.items():
        rows = power_df[(power_df["n_shots"] == int(n_shots)) & (power_df["strategy"] == strategy)].sort_values("phase_amplitude")
        rows = rows.loc[null_calibration_mask(rows)]
        if rows.empty:
            continue
        ax.errorbar(
            rows["phase_amplitude"],
            rows["power"],
            yerr=rows["power_se"],
            marker=meta["marker"],
            linewidth=1.5,
            markersize=3.2,
            markerfacecolor="white",
            markeredgewidth=0.9,
            capsize=3.0,
            capthick=1.1,
            elinewidth=1.1,
            color=meta["color"],
            zorder=5,
            label=meta["label"],
        )
    oracle_rows = power_df[(power_df["n_shots"] == int(n_shots)) & (power_df["strategy"] == ORACLE_KEY)].sort_values("phase_amplitude")
    ax.plot(oracle_rows["phase_amplitude"], oracle_rows["power"], color=ORACLE_META["color"], linestyle="-.", linewidth=2.0, label=ORACLE_META["label"])
    for target in TARGET_POWERS:
        ax.axhline(target, color="black", linestyle=":", linewidth=0.9, alpha=0.55)
    ax.set_title(rf"$n_{{\rm shot}}={int(n_shots)}$")
    ax.set_xlabel(r"Rabi phase amplitude $\Phi_R$")
    ax.set_ylabel(r"Power $1-\beta$ at $\alpha=0.05$")
    ax.set_ylim(0.0, 1.03)
    ax.legend(frameon=False, fontsize=8)

for ax in axes[plot_checkpoints.size:]:
    ax.axis("off")

fig.tight_layout()
pdf_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{RUN_PRESET}_power_curves.pdf"
fig.savefig(pdf_path, bbox_inches="tight")
print(f"saved {pdf_path}")
plt.show()


# At alpha=0.05, the stored power is exactly the probability of exceeding
# the one-sided global threshold Z = Phi_N^{-1}(1-alpha) = 1.645.
DISCOVERY_N_SHOTS = int(os.environ.get("RABI_PREFIX_DISCOVERY_N_SHOTS", str(2**14)))
if DISCOVERY_N_SHOTS not in set(power_df["n_shots"].astype(int)):
    raise ValueError(f"n_shots={DISCOVERY_N_SHOTS} is not available in the cached checkpoints")
DISCOVERY_Z = float(norm.ppf(1.0 - ALPHA))

fig, ax = plt.subplots(figsize=(6.4, 4.8))
for strategy, meta in STRATEGIES.items():
    rows = power_df[
        (power_df["n_shots"] == DISCOVERY_N_SHOTS)
        & (power_df["strategy"] == strategy)
    ].sort_values("phase_amplitude")
    rho = rows["phase_amplitude"].to_numpy(dtype=float) * math.sqrt(DISCOVERY_N_SHOTS)
    ax.errorbar(
        rho,
        rows["power"],
        yerr=rows["power_se"],
        marker=meta["marker"],
        linewidth=1.5,
        markersize=3.2,
        markerfacecolor="white",
        markeredgewidth=0.9,
        capsize=3.0,
        capthick=1.1,
        elinewidth=1.1,
        color=meta["color"],
        zorder=5,
        label=meta["label"],
    )

oracle_rows = power_df[
    (power_df["n_shots"] == DISCOVERY_N_SHOTS)
    & (power_df["strategy"] == ORACLE_KEY)
].sort_values("phase_amplitude")
oracle_rho = oracle_rows["phase_amplitude"].to_numpy(dtype=float) * math.sqrt(DISCOVERY_N_SHOTS)
ax.plot(
    oracle_rho,
    oracle_rows["power"],
    color=ORACLE_META["color"],
    linestyle="--",
    linewidth=1.8,
    label=ORACLE_META["label"],
)
ax.set_xlabel(r"Scaled signal strength $\rho=\Phi_R\sqrt{n_{\rm shot}}$")
ax.set_ylabel(r"Discovery probability $P_{H_1}(Z_{\rm global}\geq Z_\star)$")
ax.set_title(
    rf"$n_{{\rm shot}}={DISCOVERY_N_SHOTS}=2^{{{int(round(math.log2(DISCOVERY_N_SHOTS)))}}}$, "
    rf"$Z_\star={DISCOVERY_Z:.3f}$ ($\alpha={ALPHA:.2g}$)"
)
ax.set_ylim(0.0, 1.03)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
discovery_pdf_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{RUN_PRESET}_discovery_power_z{DISCOVERY_Z:.3f}.pdf"
fig.savefig(discovery_pdf_path, bbox_inches="tight")
print(f"saved {discovery_pdf_path}")
plt.show()


# Data-collapse test for the weak-signal scaling variables.
collapse_default = "256,1024,4096,16384"
COLLAPSE_SHOTS = parse_int_list(os.environ.get("RABI_PREFIX_COLLAPSE_SHOTS", collapse_default))
available_checkpoints = set(power_df["n_shots"].astype(int))
COLLAPSE_SHOTS = np.asarray([n for n in COLLAPSE_SHOTS if int(n) in available_checkpoints], dtype=int)
if COLLAPSE_SHOTS.size < 2:
    raise ValueError("the collapse plot requires at least two cached shot-count checkpoints")

collapse_colors = plt.cm.viridis(np.linspace(0.08, 0.88, COLLAPSE_SHOTS.size))
collapse_color_by_n = {int(n): color for n, color in zip(COLLAPSE_SHOTS, collapse_colors)}
collapse_panels = [
    ("fixed_x", r"Fixed $x$", 0.5, r"$\rho=\Phi_R\sqrt{n_{\rm shot}}$"),
    ("two_axis", r"Fixed $x/y$", 0.5, r"$\rho=\Phi_R\sqrt{n_{\rm shot}}$"),
    ("adaptive", "Adaptive sphere", 0.5, r"$\rho=\Phi_R\sqrt{n_{\rm shot}}$"),
    (ORACLE_KEY, "Global oracle", 0.5, r"$\rho=\Phi_R\sqrt{n_{\rm shot}}$"),
    ("fixed_z", r"Fixed $z$: correct scaling", 0.25, r"$\rho_z=\Phi_R n_{\rm shot}^{1/4}$"),
    ("fixed_z", r"Fixed $z$: transverse scaling", 0.5, r"$\rho=\Phi_R\sqrt{n_{\rm shot}}$"),
]

fig_collapse, axes_collapse = plt.subplots(2, 3, figsize=(13.2, 7.2), sharey=True)
axes_collapse = np.asarray(axes_collapse).ravel()
for ax, (strategy, title, exponent, xlabel) in zip(axes_collapse, collapse_panels):
    for n_shots in COLLAPSE_SHOTS:
        rows = power_df[
            (power_df["n_shots"] == int(n_shots))
            & (power_df["strategy"] == strategy)
        ].sort_values("phase_amplitude")
        if strategy in STRATEGIES:
            rows = rows.loc[null_calibration_mask(rows)]
        if rows.empty:
            continue
        scaled_signal = rows["phase_amplitude"].to_numpy(dtype=float) * float(n_shots) ** exponent
        color = collapse_color_by_n[int(n_shots)]
        label = rf"$n=2^{{{int(round(math.log2(n_shots)))}}}$"
        if strategy == ORACLE_KEY:
            ax.plot(scaled_signal, rows["power"], color=color, linewidth=1.7, label=label)
        else:
            ax.errorbar(
                scaled_signal,
                rows["power"],
                yerr=rows["power_se"],
                color=color,
                marker="o",
                markersize=3.0,
                markerfacecolor="white",
                markeredgewidth=0.9,
                linewidth=1.25,
                elinewidth=0.9,
                capsize=2.0,
                label=label,
            )
    ax.axhline(ALPHA, color="0.45", linestyle=":", linewidth=0.9)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"Power $1-\beta$ at $\alpha=0.05$")
    ax.set_ylim(0.0, 1.03)
    ax.legend(frameon=False, fontsize=7)

fig_collapse.tight_layout()
collapse_pdf_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{RUN_PRESET}_power_data_collapse.pdf"
fig_collapse.savefig(collapse_pdf_path, bbox_inches="tight")
print(f"saved {collapse_pdf_path}")
plt.show()


In [ ]:
ADAPTIVE_FIT_R2_MIN = float(os.environ.get("RABI_PREFIX_ADAPTIVE_FIT_R2_MIN", "0.985"))
ADAPTIVE_FIT_MIN_POINTS = int(os.environ.get("RABI_PREFIX_ADAPTIVE_FIT_MIN_POINTS", "5"))
SENSITIVITY_ERROR_DRAWS = int(os.environ.get("RABI_PREFIX_SENSITIVITY_ERROR_DRAWS", "4000"))

READOUT_SCALE = 1.0 - 2.0 * READOUT_ERROR
ETA_1 = longitudinal_relaxation_factor(SENSING_TIME, T1_TIME, T2_TIME)
ETA_2 = transverse_relaxation_factor(SENSING_TIME, T2_TIME)
P_Z0 = READOUT_ERROR + 0.5 * READOUT_SCALE * (1.0 - CONTRAST)
B_TRANSVERSE = 0.5 * READOUT_SCALE * CONTRAST * ETA_2


def solve_increasing_target(function, target: float, initial_upper: float = 1.0) -> float:
    lower = 0.0
    upper = float(initial_upper)
    while function(upper) < target and upper < 100.0:
        upper *= 2.0
    if function(upper) < target:
        return float("nan")
    return float(brentq(lambda value: function(value) - target, lower, upper, xtol=1.0e-10))


def fixed_x_required_local_snr(target: float) -> float:
    # The unknown drive phase makes the sign of the fixed-x shift unknown.
    # Average the two-sided Gaussian power over a uniform phase.
    critical = float(norm.ppf(1.0 - 0.5 * ALPHA))
    phases = np.linspace(0.0, 2.0 * np.pi, 8192, endpoint=False)
    phase_projection = np.sin(phases)

    def phase_averaged_power(local_snr: float) -> float:
        means = local_snr * phase_projection
        conditional = norm.sf(critical - means) + norm.cdf(-critical - means)
        return float(np.mean(conditional))

    return solve_increasing_target(phase_averaged_power, target, initial_upper=2.0)


def two_axis_required_noncentrality(target: float) -> float:
    critical = float(chi2.ppf(1.0 - ALPHA, df=2))
    return solve_increasing_target(
        lambda noncentrality: float(ncx2.sf(critical, df=2, nc=noncentrality)),
        target,
        initial_upper=4.0,
    )


def randomized_poisson_upper_tail_power(mu0: float, mu1: float, alpha: float) -> float:
    # Reject for K > boundary and randomize at K = boundary so that the
    # Poisson test has exactly the requested Type-I error.
    boundary = max(0, int(poisson.isf(alpha, mu0)))
    strict_null_tail = float(poisson.sf(boundary, mu0))
    boundary_null_mass = float(poisson.pmf(boundary, mu0))
    randomization = (
        np.clip((alpha - strict_null_tail) / boundary_null_mass, 0.0, 1.0)
        if boundary_null_mass > 0.0
        else 0.0
    )
    return float(poisson.sf(boundary, mu1) + randomization * poisson.pmf(boundary, mu1))


def fixed_z_poisson_sensitivity(n_shots: float, target: float) -> float:
    quadratic_response = 0.25 * READOUT_SCALE * CONTRAST * ETA_1
    mu0 = float(n_shots) * P_Z0

    def power(phase_amplitude: float) -> float:
        mu1 = float(n_shots) * (P_Z0 + quadratic_response * phase_amplitude**2)
        return randomized_poisson_upper_tail_power(mu0, mu1, ALPHA)

    return solve_increasing_target(power, target, initial_upper=0.1)


def analytic_sensitivity(strategy: str, n_values: np.ndarray, target: float) -> tuple[np.ndarray, str]:
    n_values = np.asarray(n_values, dtype=float)
    if strategy == "fixed_z":
        quadratic_response = 0.25 * READOUT_SCALE * CONTRAST * ETA_1
        gaussian_threshold = norm.ppf(1.0 - ALPHA) + norm.ppf(target)
        coefficient = math.sqrt(
            gaussian_threshold * math.sqrt(P_Z0 * (1.0 - P_Z0)) / quadratic_response
        )
        return coefficient * n_values**-0.25, rf"weak-signal analytic, $A={coefficient:.2f}$"
    if strategy == "fixed_x":
        local_snr = fixed_x_required_local_snr(target)
        coefficient = local_snr / (2.0 * B_TRANSVERSE)
        return coefficient * n_values**-0.5, rf"weak-signal analytic, $A={coefficient:.2f}$"
    if strategy == "two_axis":
        noncentrality = two_axis_required_noncentrality(target)
        coefficient = math.sqrt(noncentrality / (2.0 * B_TRANSVERSE**2))
        return coefficient * n_values**-0.5, rf"weak-signal analytic, $A={coefficient:.2f}$"
    if strategy == ORACLE_KEY:
        coefficient = (norm.ppf(1.0 - ALPHA) + norm.ppf(target)) / (2.0 * B_TRANSVERSE)
        return coefficient * n_values**-0.5, rf"weak-signal analytic, $A={coefficient:.2f}$"
    raise ValueError(f"no analytic sensitivity implemented for {strategy!r}")


def log_log_fit(n_values: np.ndarray, sensitivities: np.ndarray) -> dict:
    x = np.log(np.asarray(n_values, dtype=float))
    y = np.log(np.asarray(sensitivities, dtype=float))
    slope, intercept = np.polyfit(x, y, 1)
    prediction = intercept + slope * x
    residual = float(np.sum((y - prediction) ** 2))
    total = float(np.sum((y - np.mean(y)) ** 2))
    r_squared = 1.0 - residual / total if total > 0.0 else 1.0
    return {"intercept": float(intercept), "slope": float(slope), "r_squared": float(r_squared)}


def adaptive_tail_power_law_fit(n_values: np.ndarray, sensitivities: np.ndarray) -> dict | None:
    finite = np.isfinite(sensitivities) & (sensitivities > 0.0) & (n_values > 0.0)
    n_finite = np.asarray(n_values[finite], dtype=float)
    sensitivity_finite = np.asarray(sensitivities[finite], dtype=float)
    if n_finite.size < ADAPTIVE_FIT_MIN_POINTS:
        return None

    first_allowed = n_finite.size // 2
    last_start = n_finite.size - ADAPTIVE_FIT_MIN_POINTS
    candidates = []
    for start in range(first_allowed, last_start + 1):
        result = log_log_fit(n_finite[start:], sensitivity_finite[start:])
        result.update({"start": start, "n_values": n_finite[start:], "sensitivities": sensitivity_finite[start:]})
        candidates.append(result)

    acceptable = [candidate for candidate in candidates if candidate["r_squared"] >= ADAPTIVE_FIT_R2_MIN]
    # The first acceptable suffix is the longest well-described asymptotic range.
    selected = acceptable[0] if acceptable else max(candidates, key=lambda candidate: (candidate["r_squared"], -candidate["start"]))
    return selected


def sensitivity_crossing_interval(
    phases: np.ndarray,
    powers: np.ndarray,
    target: float,
    sample_count: int,
    draws: int,
    seed: int,
) -> tuple[float, float, float]:
    # Jeffreys-binomial draws remain non-degenerate when an observed power is
    # exactly zero or one. The null-threshold uncertainty is not included.
    successes = np.rint(np.asarray(powers, dtype=float) * sample_count).astype(int)
    rng = np.random.default_rng(seed)
    sampled_powers = rng.beta(
        successes[None, :] + 0.5,
        sample_count - successes[None, :] + 0.5,
        size=(draws, successes.size),
    )
    sampled_powers = np.maximum.accumulate(sampled_powers, axis=1)
    crossings = np.asarray(
        [sensitivity_crossing(phases, sampled, target) for sampled in sampled_powers],
        dtype=float,
    )
    finite_crossings = crossings[np.isfinite(crossings)]
    if finite_crossings.size == 0:
        return float("nan"), float("nan"), 0.0
    lower, upper = np.quantile(finite_crossings, [0.16, 0.84])
    return float(lower), float(upper), float(finite_crossings.size / crossings.size)


sensitivity_error_rows = []
for target_index, target in enumerate(TARGET_POWERS):
    for strategy_index, strategy in enumerate(STRATEGIES):
        for checkpoint_index, n_shots in enumerate(CHECKPOINTS):
            rows = power_df[
                (power_df["strategy"] == strategy)
                & (power_df["n_shots"] == int(n_shots))
            ].sort_values("phase_amplitude")
            lower, upper, finite_fraction = sensitivity_crossing_interval(
                rows["phase_amplitude"].to_numpy(dtype=float),
                rows["power"].to_numpy(dtype=float),
                float(target),
                N_SIGNAL,
                SENSITIVITY_ERROR_DRAWS,
                seed=83_000 + 10_000 * target_index + 1_000 * strategy_index + checkpoint_index,
            )
            sensitivity_error_rows.append(
                {
                    "n_shots": int(n_shots),
                    "strategy": strategy,
                    "target_power": float(target),
                    "phase_sensitivity_lower": lower,
                    "phase_sensitivity_upper": upper,
                    "crossing_draw_fraction": finite_fraction,
                }
            )

sensitivity_error_df = pd.DataFrame(sensitivity_error_rows)
sensitivity_plot_df = sensitivity_df.merge(
    sensitivity_error_df,
    on=["n_shots", "strategy", "target_power"],
    how="left",
)
calibration_df = (
    power_df[power_df["strategy"].isin(STRATEGIES)]
    .groupby(["n_shots", "strategy"], as_index=False)["empirical_alpha"]
    .first()
)
sensitivity_plot_df = sensitivity_plot_df.merge(
    calibration_df,
    on=["n_shots", "strategy"],
    how="left",
)
uncalibrated = (
    sensitivity_plot_df["strategy"].isin(STRATEGIES)
    & (
        ~np.isfinite(sensitivity_plot_df["empirical_alpha"])
        | (np.abs(sensitivity_plot_df["empirical_alpha"] - ALPHA) > NULL_CALIBRATION_TOLERANCE)
    )
)
sensitivity_plot_df.loc[
    uncalibrated,
    ["phase_sensitivity", "phase_sensitivity_lower", "phase_sensitivity_upper"],
] = np.nan


fig, axes = plt.subplots(
    1,
    len(TARGET_POWERS),
    figsize=(5.2 * len(TARGET_POWERS), 4.8),
    sharex=True,
    sharey=True,
)
axes = np.atleast_1d(axes)
fit_rows = []
analytic_rows = []

for ax, target in zip(axes, TARGET_POWERS):
    for strategy, meta in SENSITIVITY_STRATEGIES.items():
        rows = sensitivity_plot_df[(sensitivity_plot_df["strategy"] == strategy) & (sensitivity_plot_df["target_power"] == float(target))].sort_values("n_shots")
        n_values = rows["n_shots"].to_numpy(dtype=float)
        sensitivities = rows["phase_sensitivity"].to_numpy(dtype=float)
        finite = np.isfinite(sensitivities) & (sensitivities > 0.0)
        if strategy == ORACLE_KEY:
            ax.plot(
                n_values[finite],
                sensitivities[finite],
                marker=meta["marker"],
                linestyle="none",
                markersize=4.0,
                markerfacecolor="white",
                markeredgewidth=1.0,
                color=meta["color"],
                alpha=0.82,
                zorder=4,
                label=rf"{meta['label']} exact",
            )
        else:
            lower = rows["phase_sensitivity_lower"].to_numpy(dtype=float)
            upper = rows["phase_sensitivity_upper"].to_numpy(dtype=float)
            error_finite = finite & np.isfinite(lower) & np.isfinite(upper)
            yerr = np.vstack(
                [
                    np.maximum(sensitivities[error_finite] - lower[error_finite], 0.0),
                    np.maximum(upper[error_finite] - sensitivities[error_finite], 0.0),
                ]
            )
            ax.errorbar(
                n_values[error_finite],
                sensitivities[error_finite],
                yerr=yerr,
                marker=meta["marker"],
                linestyle="none",
                markersize=3.0,
                markerfacecolor="white",
                markeredgewidth=0.9,
                capsize=3.0,
                capthick=1.15,
                elinewidth=1.15,
                color=meta["color"],
                alpha=0.95,
                zorder=5,
                label=rf"{meta['label']} MC",
            )

        if strategy != "adaptive":
            dense_n = np.geomspace(float(np.min(n_values)), float(np.max(n_values)), 300)
            analytic_values, analytic_description = analytic_sensitivity(strategy, dense_n, float(target))
            analytic_label = (
                rf"{meta['label']} asymptotic $n^{{-1/4}}$"
                if strategy == "fixed_z"
                else rf"{meta['label']} analytic"
            )
            analytic_linestyle = ":" if strategy in {"fixed_x", "two_axis"} else "--"
            ax.plot(
                dense_n,
                analytic_values,
                color=meta["color"],
                linestyle=analytic_linestyle,
                linewidth=1.6,
                alpha=0.85,
                label=analytic_label,
            )
            if strategy == "fixed_z":
                poisson_values = np.asarray(
                    [fixed_z_poisson_sensitivity(n_shots, float(target)) for n_shots in dense_n],
                    dtype=float,
                )
                ax.plot(
                    dense_n,
                    poisson_values,
                    color=meta["color"],
                    linestyle=":",
                    linewidth=1.7,
                    alpha=0.95,
                    label=rf"{meta['label']} Poisson",
                )
                analytic_rows.append(
                    {
                        "target_power": float(target),
                        "strategy": "fixed_z_poisson",
                        "description": "randomized exact-alpha Poisson test with weak-signal mean",
                    }
                )
            analytic_rows.append(
                {
                    "target_power": float(target),
                    "strategy": strategy,
                    "description": analytic_description,
                }
            )
            continue

        fitted = adaptive_tail_power_law_fit(n_values, sensitivities)
        if fitted is None:
            continue
        fit_n = np.geomspace(float(fitted["n_values"][0]), float(fitted["n_values"][-1]), 160)
        fit_values = np.exp(fitted["intercept"]) * fit_n ** fitted["slope"]
        ax.plot(
            fit_n,
            fit_values,
            color=meta["color"],
            linestyle="--",
            linewidth=1.8,
            label=rf"Adaptive tail fit $s={fitted['slope']:.2f}$, $R^2={fitted['r_squared']:.3f}$",
        )
        fit_rows.append(
            {
                "target_power": float(target),
                "strategy": strategy,
                "fitted_slope": fitted["slope"],
                "intercept": fitted["intercept"],
                "r_squared": fitted["r_squared"],
                "n_fit_min": float(fitted["n_values"][0]),
                "n_fit_max": float(fitted["n_values"][-1]),
                "n_fit_points": int(fitted["n_values"].size),
            }
        )

    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xlabel(r"Number of shots $n_{\rm shot}$")
    ax.set_ylabel(r"Required $\Phi_R$")
    ax.set_title(rf"Target power $1-\beta={float(target):.2g}$")
    ax.legend(frameon=False, fontsize=7.5)

fig.tight_layout()
pdf_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{RUN_PRESET}_sensitivity.pdf"
fig.savefig(pdf_path, bbox_inches="tight")
print(f"saved {pdf_path}")
plt.show()

fit_df = pd.DataFrame(fit_rows)
analytic_df = pd.DataFrame(analytic_rows)
display(fit_df)
display(analytic_df)


In [ ]:
summary_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{RUN_PRESET}_summary.json"
csv_power_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{RUN_PRESET}_power.csv"
csv_sensitivity_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{RUN_PRESET}_sensitivity.csv"
summary = {
    "setup": setup,
    "fit_rows": fit_rows,
    "analytic_rows": analytic_rows,
    "adaptive_fit_r2_min": ADAPTIVE_FIT_R2_MIN,
    "adaptive_fit_min_points": ADAPTIVE_FIT_MIN_POINTS,
    "sensitivity_error_draws": SENSITIVITY_ERROR_DRAWS,
    "sensitivity_error_method": "Jeffreys-binomial propagation conditional on cached null thresholds",
    "cache_path": str(cache_path()),
}
summary_path.write_text(json.dumps(summary, indent=2))
power_df.to_csv(csv_power_path, index=False)
sensitivity_plot_df.to_csv(csv_sensitivity_path, index=False)
print(f"saved {summary_path}")
print(f"saved {csv_power_path}")
print(f"saved {csv_sensitivity_path}")
